# Fine-tune a Small LLM with QLoRA in Google Colab

This notebook fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` into a five-label customer-support router. It uses 4-bit quantization plus LoRA so the exercise can run on a Colab T4 GPU.

By the end, the model will return strict JSON such as:

```json
{"category": "billing"}
```

> Before running: choose **Runtime > Change runtime type > T4 GPU**.

## 1. Install and verify the environment

These versions match the environment used to test this tutorial.

In [ ]:
!pip -q install "transformers==5.17.0" "datasets==5.0.1" "accelerate==1.15.0" "peft==0.21.0" "trl==1.14.0" "bitsandbytes==0.50.2"

In [ ]:
import json
from collections import Counter

import torch
import transformers
import datasets
import accelerate
import peft
import trl
import bitsandbytes

assert torch.cuda.is_available(), (
    "No GPU detected. Select Runtime > Change runtime type > T4 GPU."
)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

## 2. Define the task and load the base model

The decision rules explicitly distinguish a past billing problem from a request to cancel future service.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
CATEGORIES = ["billing", "technical", "account", "cancellation", "shipping"]

SYSTEM_PROMPT = """
You are a customer-support routing assistant.

Classify the user's primary current problem into exactly one category:

- billing: incorrect charges, duplicate payments, invoices, refunds, or being
  charged after a subscription was already cancelled.
- technical: application, website, upload, download, crash, or software errors.
- account: login, password, verification, profile, or account security problems.
- cancellation: a request to cancel a currently active subscription, membership,
  renewal, or future service.
- shipping: orders, parcels, tracking, delivery, or damaged shipments.

Important decision rule:
If cancellation already happened and the user complains about a later charge,
classify it as billing. Use cancellation only when the user is asking to end an
active or future subscription.

Return only valid JSON in this exact format:
{"category": "category_name"}
""".strip()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)
model.eval()

print("Loaded:", MODEL_NAME)
print("Device:", model.device)
print("Approximate model memory (GB):", round(model.get_memory_footprint() / 1024**3, 2))

## 3. Build training and validation data

Each record has a conversational prompt and an ideal assistant completion. The first 12 examples per class are used for training; the final 3 are held out for validation. Hard examples teach the billing-versus-cancellation boundary.

In [ ]:
examples_by_category = {
    "billing": [
        "I was charged twice for one purchase.",
        "Why is there an extra charge on my invoice?",
        "I need a copy of my receipt.",
        "The amount on my bill is incorrect.",
        "My refund has not appeared yet.",
        "Why did my monthly price increase?",
        "I do not recognize this payment.",
        "My credit card was billed after the trial ended.",
        "Can you explain the taxes on my invoice?",
        "The discount was not applied to my bill.",
        "I was charged the wrong amount.",
        "Please send me last month's invoice.",
        "There are two identical payments on my statement.",
        "When will the refund reach my bank?",
        "The coupon did not reduce the price.",
    ],
    "technical": [
        "The application crashes when I open it.",
        "The website shows a blank screen.",
        "Uploading a file causes an error.",
        "The mobile app keeps freezing.",
        "I cannot install the latest update.",
        "The video does not load.",
        "I receive an error whenever I click submit.",
        "Notifications have stopped working.",
        "The page is loading extremely slowly.",
        "The app closes when I upload an image.",
        "The download button does nothing.",
        "The website is not displaying correctly.",
        "The program freezes during startup.",
        "I see an error code when saving a document.",
        "The application stopped working after the update.",
    ],
    "account": [
        "I forgot my password.",
        "I cannot sign in to my account.",
        "How can I change my email address?",
        "My account has been locked.",
        "I am not receiving the verification code.",
        "Someone may have accessed my account.",
        "How do I update my profile information?",
        "I need to reset my security questions.",
        "My username is not being recognized.",
        "I cannot enable two-factor authentication.",
        "Please help me recover my account.",
        "I want to change my account password.",
        "The login page says my credentials are invalid.",
        "How can I verify my email?",
        "I think someone changed my account details.",
    ],
    "cancellation": [
        "Please cancel my subscription.",
        "I want to close my membership.",
        "How do I stop my plan from renewing?",
        "Please terminate my service.",
        "I no longer want this subscription.",
        "Turn off automatic renewal.",
        "I would like to end my membership.",
        "Cancel my plan at the end of this month.",
        "How can I deactivate my paid subscription?",
        "Please close my subscription immediately.",
        "I want to discontinue the service.",
        "Do not renew my subscription next month.",
        "Please end my current plan.",
        "Where can I cancel my membership?",
        "I want to stop paying for this service.",
    ],
    "shipping": [
        "Where is my package?",
        "My parcel has not arrived.",
        "The tracking information has not updated.",
        "My order was delivered to the wrong address.",
        "The package arrived damaged.",
        "When will my order be shipped?",
        "My delivery is several days late.",
        "The courier says the package was returned.",
        "I received the wrong item in my shipment.",
        "Part of my order is missing.",
        "Can I change the delivery address?",
        "The tracking number does not work.",
        "My order still says it is being prepared.",
        "The box was damaged during delivery.",
        "When should I expect my parcel?",
    ],
}

hard_examples = [
    ("I ended my membership last week, but my card was charged today.", "billing"),
    ("My subscription is already closed, but I received another invoice.", "billing"),
    ("I cancelled the free trial, but a payment was still deducted.", "billing"),
    ("The plan was terminated last month, yet you billed me again.", "billing"),
    ("Please end my plan before the next billing date.", "cancellation"),
    ("Cancel my membership so I am not charged again.", "cancellation"),
    ("Stop the renewal before the upcoming payment.", "cancellation"),
    ("Terminate my service before the next invoice is generated.", "cancellation"),
]

In [ ]:
from datasets import Dataset

def make_record(message, category):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
        "completion": [
            {"role": "assistant", "content": json.dumps({"category": category})}
        ],
    }

train_records = []
validation_records = []

for category, messages in examples_by_category.items():
    train_records.extend(make_record(message, category) for message in messages[:12])
    validation_records.extend(make_record(message, category) for message in messages[12:])

train_records.extend(make_record(message, category) for message, category in hard_examples)

train_dataset = Dataset.from_list(train_records)
validation_dataset = Dataset.from_list(validation_records)

def label_distribution(records):
    return Counter(json.loads(record["completion"][0]["content"])["category"] for record in records)

assert len({record["prompt"][-1]["content"] for record in train_records}) == len(train_records)
assert not ({record["prompt"][-1]["content"] for record in train_records} &
            {record["prompt"][-1]["content"] for record in validation_records})

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Training distribution:", label_distribution(train_records))
print("Validation distribution:", label_distribution(validation_records))

## 4. Define evaluation and record the baseline

Exact-match accuracy requires valid JSON and the correct category.

In [ ]:
def generate_response(model_to_use, message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": message},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_to_use.device)
    with torch.inference_mode():
        outputs = model_to_use.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def evaluate_classifier(model_to_use, dataset):
    results = []
    model_to_use.eval()
    for example in dataset:
        message = example["prompt"][-1]["content"]
        expected = json.loads(example["completion"][0]["content"])["category"]
        raw_response = generate_response(model_to_use, message)
        try:
            predicted = json.loads(raw_response)["category"]
        except (json.JSONDecodeError, KeyError, TypeError):
            predicted = "INVALID_JSON"
        results.append({
            "message": message,
            "expected": expected,
            "predicted": predicted,
            "raw_response": raw_response,
            "correct": predicted == expected,
        })
    accuracy = sum(result["correct"] for result in results) / len(results)
    return accuracy, results

In [ ]:
baseline_accuracy, baseline_results = evaluate_classifier(model, validation_dataset)
print(f"Base-model validation accuracy: {baseline_accuracy:.1%}")
for result in baseline_results:
    print(result["expected"], "<-", result["predicted"], "|", result["message"])

## 5. Configure QLoRA and train

The base weights remain quantized and frozen. Only the small LoRA adapter parameters are updated.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_config = SFTConfig(
    output_dir="/content/support-router-training",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    max_length=256,
    completion_only_loss=True,
    packing=False,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

model.config.use_cache = False
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# Avoid a BF16 gradient-unscale mismatch on Tesla T4.
for parameter in trainer.model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float32:
        parameter.data = parameter.data.to(torch.float32)

trainer.model.print_trainable_parameters()
print("Trainable dtypes:", Counter(
    str(parameter.dtype) for parameter in trainer.model.parameters() if parameter.requires_grad
))

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

## 6. Evaluate the trained adapter

Validation examples were not used for gradient updates. Boundary tests additionally check the most confusing label distinction.

In [ ]:
trainer.model.config.use_cache = True
trainer.model.eval()

trained_accuracy, trained_results = evaluate_classifier(trainer.model, validation_dataset)
print(f"Before training: {baseline_accuracy:.1%}")
print(f"After training:  {trained_accuracy:.1%}")

for result in trained_results:
    symbol = "✅" if result["correct"] else "❌"
    print(symbol, result["expected"], "<-", result["predicted"], "|", result["message"])

In [ ]:
boundary_cases = [
    ("Cancel my subscription so I am not charged next month.", "cancellation"),
    ("I was charged even though I cancelled last month.", "billing"),
    ("You billed me again after confirming my cancellation.", "billing"),
    ("Please end my plan before the next payment.", "cancellation"),
    ("A payment was taken after my plan had already ended.", "billing"),
    ("Stop my plan before the next billing date.", "cancellation"),
]

boundary_correct = 0
for message, expected in boundary_cases:
    raw = generate_response(trainer.model, message)
    try:
        predicted = json.loads(raw)["category"]
    except (json.JSONDecodeError, KeyError, TypeError):
        predicted = "INVALID_JSON"
    is_correct = predicted == expected
    boundary_correct += int(is_correct)
    print(("✅" if is_correct else "❌"), expected, "<-", predicted, "|", message)

print(f"Boundary accuracy: {boundary_correct}/{len(boundary_cases)}")

In [ ]:
test_messages = [
    "I was charged twice for the same order.",
    "The application crashes whenever I upload a photo.",
    "Please close my subscription immediately.",
    "My parcel has still not arrived.",
    "I forgot my password and cannot sign in.",
]

for message in test_messages:
    print("MESSAGE:", message)
    print("PREDICTION:", generate_response(trainer.model, message))
    print("-" * 70)

## 7. Save the LoRA adapter

The first cell saves to the temporary Colab VM. Set `SAVE_TO_DRIVE = True` in the following cell if you also want a persistent Google Drive copy. The full base model is not saved; inference later requires both the base model and this adapter.

In [ ]:
import os

LOCAL_SAVE_DIR = "/content/support-router-lora"
trainer.model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)

with open(os.path.join(LOCAL_SAVE_DIR, "task_config.json"), "w", encoding="utf-8") as file:
    json.dump({
        "base_model": MODEL_NAME,
        "categories": CATEGORIES,
        "system_prompt": SYSTEM_PROMPT,
    }, file, indent=2)

print("Saved locally:", LOCAL_SAVE_DIR)
print(os.listdir(LOCAL_SAVE_DIR))

In [ ]:
SAVE_TO_DRIVE = False  # Change to True and run this cell to persist the adapter.

if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/llm-finetuning/support-router-lora"
    os.makedirs(os.path.dirname(DRIVE_SAVE_DIR), exist_ok=True)
    shutil.copytree(LOCAL_SAVE_DIR, DRIVE_SAVE_DIR, dirs_exist_ok=True)
    print("Saved to Google Drive:", DRIVE_SAVE_DIR)
else:
    print("Drive save skipped. Set SAVE_TO_DRIVE = True to enable it.")

## What you learned

- QLoRA reduces memory by quantizing the frozen base model while training small adapters.
- Fine-tuning teaches task behavior and label boundaries; it is not a replacement for frequently changing knowledge retrieval.
- Held-out tests and hard boundary examples are essential because low training loss alone does not prove generalization.
- This synthetic dataset is appropriate for learning, not for claiming production performance.